In [ ]:
import numpy as np
np.random.seed(42)
readings = np.random.normal(35, 15, (8, 24))
readings = np.round(readings, 1)
readings[2, 5] = -999  # error on the 3rd sensor the 6th hour reading
readings[6, 18] = np.nan  # not a number at the 7th sensor, the 19th hour
readings[0, 12] = 250.0  # spike in readings maybe a fire took place

# find error codes and any outliers in the matrix readings
errors = readings == -999  # for every single line in the matrix is it equal to -999
# replace error reading
readings[readings == -999] = np.nan  # if any value equals -999 replace it with nan

# find missing values (NaN)
missing = np.isnan(readings)
# print out a message
print(f'Total NaN detected: {missing.sum()}')  # sums up the number of True values from the mask
# replace the missing values
np.any(missing, axis=1)  # checks if any values are True, check row wise

# cap any outliers
spikes = readings > 200
# how many spiked readings do we have?
print(f'Total no of spikes: {spikes.sum()}')  # fixed: was spike.sum()
# applying the mask to cap the readings
readings[readings > 200] = 200

# summary of the clean up
clean_count = (np.isnan(readings)).sum()
print(f'Clean readings: {clean_count}')

# rank the sensors by mean
means = np.nanmean(readings, axis=1)  # fixed: was np.namean, also fixed variable name to means
ranked = np.argsort(means)[::-1]  # the highest mean sensor readings means unhealthiest
top3 = ranked[:3]  # the highest readings of pollution

# reshape the matrix (8, 24) — rows=sensors, cols=hours
reshape = readings.T  # shape (24, 8) — rows=hours, cols=sensors
# hourly mean
hourly_mean = np.nanmean(readings, axis=1)  # fixed: was np.namean and reading

# how many std away is the sensor from its average
mea = np.nanmean(readings, axis=1)  # fixed: was np.namean, missing comma
std = np.nanstd(readings, axis=1)  # std of each sensor (8,)

# in order to do value - mean the shapes of the array need to be the same
mea = mea.reshape(-1, 1)  # now we can subtract (8,24)-(8,1)
std = std.reshape(-1, 1)  # (8,1) for math operations
# z score - what a reading means relative to the sensor's normal based off mean and std
# z = readings - mean / std
z = (readings - mea) / std  # fixed: was mu which didn't exist
max_z_value = np.nanmax(z)
rows, cols = np.where(z == max_z_value)

# summary report
daily_mean = np.nanmean(readings, axis=1)  # fixed: was sensors which didn't exist
daily_max = np.nanmax(readings, axis=1)    # fixed: was sensors
daily_std = np.nanstd(readings, axis=1)    # fixed: was sensors

# already cleaned
readings[readings < 0] = 0.0  # any readings less than 0 are 0.0
readings = np.clip(readings, 0, 200)

# what is each sensor's peak across the 24 hours
peak = np.maximum.accumulate(readings, axis=1)  # biggest value seen in a 24 hour period

# total pollution per sensor
total_pollution = np.add.reduce(readings, axis=1)
print(total_pollution)  # total pollution per sensor

# distance for every possible pair
positions = np.random.rand(8, 2) * 10  # sensors and their 2 coords x and y of a 10 by 10 km grid

diff = positions[:, np.newaxis, :] - positions[np.newaxis, :, :]
dist_matrix = np.sqrt((diff**2).sum(axis=2))  # translates our position vectors to a single number
print(f'distance from sensor 0 to sensor 5: {dist_matrix[0, 5]} km')

# assigning aqi index scores
conditions = [
    readings > 150,
    readings > 75,
    readings > 35,
]
choices = [4, 3, 2]  # fixed: was strings '4','3','2' — can't average strings
aqi_scores = np.select(conditions, choices, default=1)  # fixed: default was '1'
print(f'Mean aqi per sensor: {aqi_scores.mean(axis=1)}')

# print out the worst pollution hours
alert_hours = (aqi_scores >= 3)
highest_hours = readings[alert_hours]
print(f'The following times are when pollution is highest: {highest_hours}')

# hourly changes
hourly_change = np.diff(readings, axis=1)
biggest_spike = np.max(hourly_change, axis=1)
spike_hour = np.argmax(biggest_spike)  # fixed: argmax on 1D returns scalar, not array
print(f'Biggest spike: Sensor {np.argmax(biggest_spike)}, +{biggest_spike.max():.1f} at hour {spike_hour}')

# ranking the best performing sensors
best_hours = (aqi_scores < 3)  # fixed: was aqi which didn't exist
lowest_readings = readings[best_hours]
daily_mean = readings.mean(axis=1)
rank = np.lexsort((daily_mean, aqi_scores.mean(axis=1)))  # fixed: lexsort takes a tuple

# printing the sensors in the format of rank: sensor
for pos, sensor in enumerate(rank):  # fixed: was idx which didn't exist
    print(f" #{pos+1} Sensor {sensor}: alerts={alert_hours[sensor].sum()}h, "
          f"mean={daily_mean[sensor]:.1f}")

# save cleaned sensor data
# saving the sensor data and the means of each sensor in a compressed file
np.savez_compressed('busan_clean_air.npz', sensors=readings,
                    daily_mean=readings.mean(axis=1))
# human readable file in csv format
np.savetxt('busan_clean_air.csv', readings, delimiter=',')  # fixed: was np.save with wrong args

# simulate future predictions with the current data we have
# we are trying to find the line of best fit for a set of data points
# y = mx + c where our readings are y and x is our number of hours
# we use np.linalg.lstsq to find the slope and the intercept c

hours = np.arange(24)
ones = np.ones(24)  # fixed: was missing closing parenthesis

for i in range(8):
    A = np.column_stack([hours, ones])
    parameters = np.linalg.lstsq(A, readings[i], rcond=None)[0]  # fixed: was lstq and paremeter
    slope = parameters[0]  # fixed: was paremeter
    intercept = parameters[1]
    print(f" reading{i}: slope={slope:+.2f}/hr, intercept={intercept:.1f}")
